# Odwołania2json – walkthrough

This notebook walks through the solution step by step, using the project's own modules from `src/`:

1. **Text** – Docling turns a PDF into text blocks (OCR only where there is no text layer).
2. **Document type** – weighted keywords from `config/doc_types/*/doc_type.toml`.
3. **Fields** – rules from the same config fill the output structure defined by `schema.json`.
4. **LLM** – small tasks the rules cannot settle (the appeal's reason code, names in the nominative case), Bielik-11B through an EU-hosted API.
5. **Provenance** – every value records the rule or LLM task, page and text it came from.

Then it runs the whole pipeline on a mixed set of documents, shows how a document type is configured without code, and evaluates the solution on all 226 documents provided by the organiser.

It is saved with its outputs, so it can be read without running it. To run it: install the project (see `README.md`), start Jupyter in the repo root (`python -m notebook walkthrough.ipynb`) and run all cells. Without `LLM_API_KEY` in `.env`, the LLM steps are skipped and rules alone are used. Nothing is written to `output/` or `eval/reports/`; the notebook works in a temporary folder.

In [1]:
import json
import logging
from dataclasses import replace
from pathlib import Path
from tempfile import mkdtemp

import pandas as pd
from IPython.display import Markdown, display

import src.run as run
from src.cache import DiskCache
from src.classification.keyword_classifier import KeywordClassifier
from src.doc_types import load_doc_types
from src.experiments.experiment import select_experiments
from src.experiments.reports import ReportStore
from src.experiments.runner import ExperimentRunner
from src.extraction.rule_extractor import RuleExtractor
from src.inference.llm_client import LlmClient
from src.pipeline.documents import InputDocument
from src.text_extraction.text_extractor import DoclingTextExtractor

run.configure_logging(logging.WARNING)  # keep the notebook output to results
run.load_env_file(run.ENV_FILE)
pd.set_option("display.max_colwidth", 90)

scratch = Path(mkdtemp(prefix="odwolania2json-"))
workspace = replace(run.WORKSPACE, output_dir=scratch / "output", traces_dir=scratch / "traces",
                    runs_dir=scratch / "runs", reports_dir=scratch / "reports")
[hybrid, rules] = select_experiments(run.EXPERIMENTS, ["hybrid-v1", "rules-v1"])
config = hybrid.config
llm_available = config.llm.api_key is not None
doc_types = load_doc_types(workspace.doc_types_dir)

print("Document types:", ", ".join(doc_types))
print("LLM:", config.llm.model if llm_available else "no LLM_API_KEY - LLM steps fall back to rules")


def sources_table(sources):
    """The provenance of extracted values as a table."""
    return pd.DataFrame([{"field": s.path, "value": s.value, "source": s.source, "rule / task": s.detail,
                          "page": s.page, "text": (s.snippet or "")[:90]} for s in sources])

Document types: odwolanie, pelnomocnictwo, przelew
LLM: Bielik-11B-v3.0-Instruct


## Step 1 – Text extraction

A realistic appeal: 11 pages, of which the first 2 are converted (every value the references expect is on pages 1–2). Conversions are cached by file content and settings; the cold conversion time is stored with the cache entry.

In [2]:
sample = Path("dane/przyklad_47/odwolanie.pdf")
extractor = DoclingTextExtractor(config.conversion, workspace.models_dir, DiskCache(workspace.cache_dir / "docling"))
document = extractor.extract(sample)
print(f"{document.page_count} pages in the PDF, {config.conversion.max_pages} converted, "
      f"{len(document.blocks)} text blocks, cold conversion {document.conversion_s:.2f} s")
print()
print(document.text[:1500])

11 pages in the PDF, 2 converted, 48 text blocks, cold conversion 0.45 s

Krajowa Izba Odwoławcza
ul. Postępu 17A
02-676 Warszawa Poznań, 28.08.2026
Odwołujący:
RAVELO-BUD Sebastian Tomaszewski
ul. Jagiellońska 91, 70-402 Szczecin
NIP: 9997975723, REGON: 999544334
przedsiębiorca wpisany do CEIDG, prowadzący działalność gospodarczą pod firmą
RAVELO-BUD Sebastian Tomaszewski
reprezentowany przez: adw. Elżbieta Woźniak, r. pr. Iwona Kucharska
adres do korespondencji: Kancelaria Radcy Prawnego Elżbieta Woźniak, ul. Ogrodowa 38,
61-732 Poznań
tel. 000 000 000, e-mail: kontakt@wozniakkancelaria.example
Zamawiający:
Miejskie Przedsiębiorstwo Komunikacyjne w Pogorzeli Wielkiej sp. z o.o.
ul. Sienkiewicza 57, 22-403 Pogorzel Wielka
KRS: 9998553316, NIP: 9994443606, REGON: 999257689
przetargi@pogorzelwi.example
dotyczy: postępowania o udzielenie zamówienia publicznego prowadzonego w trybie podstawowym bez negocjacji pn. "Świadczenie usług przewozu uczniów do szkół w roku szkolnym 2026/2027", nr 

## Step 2 – Document type

Each type scores the summed weights of its keywords (`doc_type.toml`); ruling markers such as „wyrok” count against an appeal only at the top of the document. The best type wins if it reaches its `min_score`; anything else gets `{}`.

In [3]:
classification = KeywordClassifier(doc_types, config.classification).classify(document)
display(pd.DataFrame({"score": classification.scores,
                      "min_score": {name: t.classification.min_score for name, t in doc_types.items()}}))
print("Document type:", classification.doc_type)

,score,min_score
odwolanie,6.0,4.0
pelnomocnictwo,1.0,3.0
przelew,0.0,3.0


Document type: odwolanie


## Step 3 – Field extraction with rules

The rules in `config/doc_types/odwolanie/doc_type.toml` fill the structure from `schema.json`; values are normalised by their value type (ISO dates, digits-only identifiers, amounts as numbers). Below: the output JSON and where each value came from.

In [4]:
doc_type = doc_types[classification.doc_type]
by_rules = RuleExtractor(doc_types).extract(document, doc_type)
print(json.dumps(by_rules.data, ensure_ascii=False, indent=2))

{
  "OdwolanieKIO": {
    "data": "2026-08-28",
    "czynnosc_dzien": "2026-08-23",
    "czynnosc_powod_id": 8,
    "czynnosc_opis": "Zaniechanie wykluczenia wykonawcy",
    "miejscowosc": "Poznań"
  },
  "Odwolujacy": [
    {
      "krs": null,
      "nazwa": "RAVELO-BUD Sebastian Tomaszewski",
      "nip": "9997975723",
      "regon": "999544334",
      "przez-pelnomocnika": 1,
      "pelnomocnik": {
        "imie_nazwisko": "Elżbieta Woźniak",
        "nazwa": "Kancelaria Radcy Prawnego Elżbieta Woźniak",
        "mail": "kontakt@wozniakkancelaria.example"
      }
    }
  ],
  "Zamawiajacy": [
    {
      "krs": "9998553316",
      "nazwa": "Miejskie Przedsiębiorstwo Komunikacyjne w Pogorzeli Wielkiej sp. z o.o.",
      "nip": "9994443606",
      "regon": "999257689",
      "prowadzacy_postepowanie": null
    }
  ],
  "ZamowieniePubliczne": {
    "numer_bzp": "2026/BZP00999143",
    "numer_postepowania": "ZP.271/130/2026",
    "numer_ted": "999583-2026",
    "tytul": "Świadczenie us

In [5]:
sources_table(by_rules.sources)

,field,value,source,rule / task,page,text
0,OdwolanieKIO.data,2026-08-28,rule,"label \b[a-z][a-z-]+,\s*(?:dnia\s*)?(?=\d)",1,"dwoławcza ul. Postępu 17A 02-676 Warszawa Poznań, 28.08.2026 Odwołujący: RAVELO-BUD Se..."
1,OdwolanieKIO.miejscowosc,Poznań,rule,pattern in section,1,"a Izba Odwoławcza ul. Postępu 17A 02-676 Warszawa Poznań, 28.08.2026 Odwołujący: RAVEL..."
2,OdwolanieKIO.czynnosc_dzien,2026-08-23,rule,first in section,2,ostępowanie nr ZP.271/130/2026 Zamawiający w dniu 23 sierpnia 2026 roku przekazał info...
3,OdwolanieKIO.czynnosc_powod_id,8,rule,classify 8: zaniecha\w* wykluczeni,2,"rmację o wyniku postępowania, z której wynika, że zaniechał wykluczenia wykonawcy Wist..."
4,OdwolanieKIO.czynnosc_opis,Zaniechanie wykluczenia wykonawcy,derived,lookup by OdwolanieKIO.czynnosc_powod_id,2,"rmację o wyniku postępowania, z której wynika, że zaniechał wykluczenia wykonawcy Wist..."
5,Odwolujacy[0].nazwa,RAVELO-BUD Sebastian Tomaszewski,rule,first in section,1,"7A 02-676 Warszawa Poznań, 28.08.2026 Odwołujący: RAVELO-BUD Sebastian Tomaszewski ul...."
6,Odwolujacy[0].nip,9997975723,rule,label nip\b,1,"szewski ul. Jagiellońska 91, 70-402 Szczecin NIP: 9997975723, REGON: 999544334 przedsi..."
7,Odwolujacy[0].regon,999544334,rule,label regon\b,1,"ońska 91, 70-402 Szczecin NIP: 9997975723, REGON: 999544334 przedsiębiorca wpisany do ..."
8,Odwolujacy[0].przez-pelnomocnika,1,rule,classify 1: reprezentowan\w* przez,1,darczą pod firmą RAVELO-BUD Sebastian Tomaszewski reprezentowany przez: adw. Elżbieta ...
9,Odwolujacy[0].pelnomocnik.imie_nazwisko,Elżbieta Woźniak,rule,label reprezentowan\w* przez(?:\s+p\w+nomocnika)?\s*:?,1,"Sebastian Tomaszewski reprezentowany przez: adw. Elżbieta Woźniak, r. pr. Iwona Kuchar..."


## Step 4 – LLM tasks

The rules guess the appeal's reason code from patterns in the sentence „Zamawiający w dniu … <action>.”. The `contested_action` task (defined in the same config) gives that sentence to Bielik-11B with the 12 codes; a valid answer replaces the rule value and the description is derived again. Any LLM failure keeps the rule value.

For the appeal above, rules and LLM agree. Here is another appeal, where the patterns pick "publication of the SWZ" (4) but the contested action is the answers to contractors' questions (6):

In [6]:
llm_sample = Path("dane/przyklad_44/odwolanie.pdf")
appeal = extractor.extract(llm_sample)
by_rules = RuleExtractor(doc_types).extract(appeal, doc_type)
if llm_available:
    llm = LlmClient(config.llm, DiskCache(workspace.cache_dir / "llm"))
    with_llm = RuleExtractor(doc_types, llm).extract(appeal, doc_type)
    rule_values = {s.path: s.value for s in by_rules.sources}
    changed = [s for s in with_llm.sources if s.source == "llm" or rule_values.get(s.path) != s.value]
    display(pd.DataFrame([{"field": s.path, "rules": rule_values.get(s.path), "with LLM": s.value,
                           "source": s.source, "rule / task": s.detail} for s in changed]))
    print("Expected (reference):", json.loads(llm_sample.with_suffix(".json").read_text())["OdwolanieKIO"]["czynnosc_powod_id"])
    print("Text around it:", next(s.snippet for s in with_llm.sources if s.source == "llm"))
else:
    print("No LLM_API_KEY: skipped (rules alone are used).")

,field,rules,with LLM,source,rule / task
0,OdwolanieKIO.czynnosc_powod_id,4,6,llm,task contested_action
1,OdwolanieKIO.czynnosc_opis,Publikacja specyfikacji warunków zamówienia (SWZ),Odpowiedzi/wyjaśnienia do SWZ,derived,lookup by OdwolanieKIO.czynnosc_powod_id


Expected (reference): 6
Text around it: ul. Postępu 17A 02-676 Warszawa Zamawiający w dniu 9 marca 2026 roku udzielił odpowiedzi na pytania wykonawców (pytania nr 14-27) i opublikował wyjaśnienia treści SWZ na stronie internetowej prowadzonego postępowania. Następnie, w dniu 11 marca 2026 r., Zamawiający


## Step 5 – The whole pipeline on a mixed set

The same pipeline the CLI (`python -m src.run`), the API and the web UI use: an appeal, a power of attorney and a transfer confirmation of one case, a scanned power of attorney (OCR path) and a procurement notice that must not be recognised.

In [7]:
samples = [Path("dane/przyklad_47/odwolanie.pdf"), Path("dane/przyklad_47/pelnomocnictwo.pdf"),
           Path("dane/przyklad_47/potwierdzenie_przelewu.pdf"), Path("dane/skany/pelnomocnictwa/0003/0003.pdf"),
           Path("dane/negatywne/2025_S 112-383751.pdf")]
pipeline = (hybrid if llm_available else rules).build_pipeline(workspace)
results = list(pipeline.run([InputDocument(case=path.parent.name, path=path) for path in samples]))
pd.DataFrame([{"file": str(r.document.path.relative_to("dane")), "type": r.doc_type or "{} (not recognised)",
               "values": len(r.sources), "from LLM": sum(s.source == "llm" for s in r.sources),
               "conversion s (cold)": r.extracted.conversion_s if r.extracted else None} for r in results])

,file,type,values,from LLM,conversion s (cold)
0,przyklad_47/odwolanie.pdf,odwolanie,20,1,0.452
1,przyklad_47/pelnomocnictwo.pdf,pelnomocnictwo,7,0,0.237
2,przyklad_47/potwierdzenie_przelewu.pdf,przelew,6,0,0.611
3,skany/pelnomocnictwa/0003/0003.pdf,pelnomocnictwo,6,0,1.981
4,negatywne/2025_S 112-383751.pdf,{} (not recognised),0,0,0.781


In [8]:
transfer = next(r for r in results if r.doc_type == "przelew")
print(json.dumps(transfer.data, ensure_ascii=False, indent=2))
sources_table(transfer.sources)

{
  "przelew_bankowy": {
    "data_operacji": "2026-08-27",
    "kwota": 100.0,
    "na_rachunek": "33237021642224014358154704",
    "numer_transakcji": "CEN2608281325159",
    "tytul": "opłata skarbowa od pełnomocnictwa dla Elżbieta Woźniak i Iwona Kucharska w postępowaniu przed KIO w przetargu ZP.271/130/2026 CEN2608281325159",
    "z_rachunku": "22147553851780849966673534"
  }
}


,field,value,source,rule / task,page,text
0,przelew_bankowy.z_rachunku,22147553851780849966673534,rule,label rachunek zleceniodawcy,1,"ńska 91, 70-402 Szczecin Rachunek zleceniodawcy | PL22147553851780849966673534 Odbiorc..."
1,przelew_bankowy.na_rachunek,33237021642224014358154704,rule,label rachunek odbiorcy,1,"za 57, 22-403 Pogorzel Wielka Rachunek odbiorcy | PL33237021642224014358154704 Kwota |..."
2,przelew_bankowy.kwota,100.0,rule,label kwota,1,"k odbiorcy | PL33237021642224014358154704 Kwota | 100,00 PLN Data operacji | 2026-08-2..."
3,przelew_bankowy.data_operacji,2026-08-27,rule,label data operacji,1,"24014358154704 Kwota | 100,00 PLN Data operacji | 2026-08-27 Tytuł przelewu | opłata s..."
4,przelew_bankowy.numer_transakcji,CEN2608281325159,rule,label numer referencyjny,1,71/130/2026 CEN2608281325159 Numer referencyjny | CEN2608281325159 Status | ZREALIZOWA...
5,przelew_bankowy.tytul,opłata skarbowa od pełnomocnictwa dla Elżbieta Woźniak i Iwona Kucharska w postępowani...,rule,label tytul przelewu,1,0 PLN Data operacji | 2026-08-27 Tytuł przelewu | opłata skarbowa od pełnomocnictwa dl...


## Step 6 – A document type is configuration only

Everything specific to a type lives in `config/doc_types/<type>/`: `schema.json` (the output shape, with a value type in every leaf) and `doc_type.toml` (keywords, sections, field rules, LLM tasks). Adding a type needs no code – see `docs/extensibility.md`. The transfer type:

In [9]:
folder = workspace.doc_types_dir / "przelew"
print((folder / "schema.json").read_text(encoding="utf-8"))
toml = (folder / "doc_type.toml").read_text(encoding="utf-8")
print(toml[toml.index('[fields."przelew_bankowy.kwota"]'):toml.index('[fields."przelew_bankowy.numer_transakcji"]')])

{
  "przelew_bankowy": {
    "data_operacji": "date",
    "kwota": "amount",
    "na_rachunek": "account",
    "numer_transakcji": "text",
    "tytul": "text",
    "z_rachunku": "account"
  }
}

[fields."przelew_bankowy.kwota"]
labels = ["kwota", "w kwocie", "na kwote", "wartosc"]
window = 120

[fields."przelew_bankowy.data_operacji"]
labels = ["data operacji", "data transakcji", "data realizacji", "data wykonania", "data obciazenia",
          "data ksiegowania", "data zlecenia", "w dniu", "data waluty", 'data\b']
window = 120




## Step 7 – Evaluation on all 226 documents

The organiser's data: 184 synthetic documents (60 cases, some with negatives inside), 12 negatives and 30 scans. A PDF's reference is the JSON with the same name next to it; a PDF without one must produce `{}`. F1 compares every value with the reference (equal → TP, different → FP + FN, missing → FN).

On a warm cache this takes seconds; cold, it converts all 226 PDFs (a few minutes) and makes 70 LLM calls.

In [10]:
experiments = [rules, hybrid] if llm_available else [rules]
reports = {e.name: ExperimentRunner(workspace, run.EVAL_SETS).run(e) for e in experiments}
best = reports[experiments[-1].name]

overview = []
for name, report in reports.items():
    s = report.scores
    row = {"variant": name, "F1": s.strict.f1, "precision": s.strict.precision, "recall": s.strict.recall,
           "classification": s.classification.accuracy}
    row |= {f"F1 {subset}": scores.strict.f1 for subset, scores in s.by_subset.items() if subset != "negatives"}
    row["invented values on negatives"] = s.by_subset["negatives"].strict.fp
    overview.append(row)
pd.DataFrame(overview).set_index("variant").round(3)

,F1,precision,recall,classification,F1 synthetic,F1 scans,invented values on negatives
variant,,,,,,,
rules-v1,0.950,0.967,0.934,1.0,0.969,0.822,0
hybrid-v1,0.954,0.970,0.938,1.0,0.973,0.822,0


In [11]:
s = best.scores
by_type = pd.DataFrame({t: {"F1": v.f1, "TP": v.tp, "FP": v.fp, "FN": v.fn} for t, v in s.by_doc_type.items() if t != "none"}).T
display(Markdown(f"**{best.run.experiment}** by document type:"), by_type.round(3))
fields = pd.DataFrame({path: {"F1": v.f1, "TP": v.tp, "FP": v.fp, "FN": v.fn,
                              "values in text": s.coverage_by_field[path].share if path in s.coverage_by_field else None}
                       for path, v in s.by_field.items()}).T.sort_values("F1")
display(Markdown("Weakest fields (*values in text* = share of reference values present in the extracted text, the ceiling for extraction):"),
        fields.head(10).round(3))

**hybrid-v1** by document type:

,F1,TP,FP,FN
odwolanie,0.965,1323.0,29.0,68.0
pelnomocnictwo,0.922,481.0,27.0,55.0
przelew,0.958,391.0,11.0,23.0


Weakest fields (*values in text* = share of reference values present in the extracted text, the ceiling for extraction):

,F1,TP,FP,FN,values in text
Zamawiajacy[].prowadzacy_postepowanie,0.000,0.0,0.0,8.0,NaN
mocodawca[].podpisujacy[].imie_nazwisko,0.820,91.0,22.0,18.0,1.000
Odwolujacy[].nazwa,0.900,63.0,7.0,7.0,0.986
Zamawiajacy[].nazwa,0.908,64.0,3.0,10.0,1.000
mocodawca[].data.nazwa,0.918,67.0,3.0,9.0,1.000
OdwolanieKIO.czynnosc_opis,0.921,64.0,5.0,6.0,NaN
OdwolanieKIO.czynnosc_powod_id,0.921,64.0,5.0,6.0,NaN
mocodawca[].data.krs,0.926,63.0,1.0,9.0,0.972
przelew_bankowy.tytul,0.928,64.0,4.0,6.0,0.914
przelew_bankowy.z_rachunku,0.929,65.0,5.0,5.0,1.000


The weakest fields have known causes: `prowadzacy_postepowanie` is set only when an appeal names several contracting authorities (2 of 60 appeals; the rules take the first), signatories of consortia need multi-entity extraction, and on scans OCR errors (lost Polish diacritics) limit what is in the text at all.

## Step 8 – Time budget and model choice

The organiser allows 45 minutes for 150 PDFs on a CPU-only machine. Cold conversion times per document (stored with every cached conversion) and the LLM cost:

In [12]:
timing = pd.DataFrame({subset: {"mean s / document": v.conversion_s, "max s": v.max_conversion_s}
                       for subset, v in best.scores.by_subset.items()}).T
display(timing)
for subset in ("synthetic", "scans"):
    print(f"150 PDFs like the {subset} set: about {timing.loc[subset, 'mean s / document'] * 150 / 60:.1f} min of conversion")
if usage := best.run.llm_usage:
    print(f"LLM: {usage.prompt_tokens + usage.completion_tokens} tokens in total "
          f"({(usage.prompt_tokens + usage.completion_tokens) / best.run.documents:.0f} per document), {usage.seconds:.0f} s on a cold run")

,mean s / document,max s
synthetic,0.42,0.92
negatives,1.75,5.38
scans,3.43,7.09


150 PDFs like the synthetic set: about 1.1 min of conversion
150 PDFs like the scans set: about 8.6 min of conversion
LLM: 41072 tokens in total (182 per document), 22 s on a cold run


The LLM tasks were evaluated with four open models on the same EU-hosted endpoint (committed reports in `eval/reports/`). Bielik-11B, the smallest dense model, is also the most accurate, so it is the default:

In [13]:
latest = {report.run.experiment: report for report in ReportStore(run.WORKSPACE.reports_dir).load()}
models = {"rules-v1": "no LLM", "hybrid-v1": "Bielik-11B (11B)", "hybrid-mistral": "Mistral-Small-3.2 (24B)",
          "hybrid-gemma": "gemma-4-26B-A4B (26B MoE, 4B active)", "hybrid-qwen": "Qwen3.8-27B (27B)"}
rows = []
for name, model in models.items():
    report = latest[name]
    usage = report.run.llm_usage
    rows.append({"variant": name, "model": model, "F1": report.scores.strict.f1,
                 "reason code F1": report.scores.by_field["OdwolanieKIO.czynnosc_powod_id"].f1,
                 "LLM tokens / document": round((usage.prompt_tokens + usage.completion_tokens) / report.run.documents) if usage else 0,
                 "unusable LLM answers": usage.failures if usage else 0})
pd.DataFrame(rows).set_index("variant").round(3)

,model,F1,reason code F1,LLM tokens / document,unusable LLM answers
variant,,,,,
rules-v1,no LLM,0.950,0.863,0,0
hybrid-v1,Bielik-11B (11B),0.954,0.921,182,0
hybrid-mistral,Mistral-Small-3.2 (24B),0.952,0.892,139,0
hybrid-gemma,"gemma-4-26B-A4B (26B MoE, 4B active)",0.948,0.834,133,0
hybrid-qwen,Qwen3.8-27B (27B),0.952,0.892,255,17


## Summary

- **Accuracy**: F1 0.954 on the organiser's 226 documents (synthetic 0.973, scans 0.822), document type 100% correct, no invented values on negatives; 0.950 with rules alone.
- **Small LLM**: Bielik-11B for one question per appeal (~180 tokens per document); without an API key the solution still runs on rules alone.
- **Configuration over code**: document types, rules and LLM tasks live in `config/doc_types/`; a new type needs no code.
- **Transparency**: every value records its source – rule or LLM task, page and text (`traces/`, the API and the web UI).
- **Time**: about 1 minute of conversion for 150 text PDFs, under 9 minutes if all are scans – well within 45 minutes.

Run it yourself: `python -m src.run` (folder → JSON), `python -m uvicorn src.api:app` (API with Swagger at `/docs`), `python -m streamlit run src/web_app.py` (web UI). Details in `README.md` and `docs/` (architecture, extensibility, evaluation, infrastructure).